In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load the local embedding model
local_model_path = "../models/all-MiniLM-L6-v2/"
embeddings = HuggingFaceEmbeddings(model_name=local_model_path)

# 2. Load the FAISS index
vector_db = FAISS.load_local(
    "../vector_store/complaints_index", 
    embeddings, 
    allow_dangerous_deserialization=True
)

# 3. Define the Retriever (k=5)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

print("✅ Retriever is ready and pointing to your local 'Brain'!")

C:\Users\DELL 7020\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\DELL 7020\AppData\Local\Temp\ipykernel_13464\3775375851.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5125.02it/s]


✅ Retriever is ready and pointing to your local 'Brain'!


In [ ]:
import sys
import os

# 1. Force install the specific missing modules
!{sys.executable} -m pip install -U langchain langchain-community langchain-core langchain-groq

print(f"Python Executable: {sys.executable}")
print("--- Check if LangChain exists ---")
try:
    import langchain
    print(f"LangChain Location: {langchain.__file__}")
except ImportError:
    print("LangChain still not found")

# 2. Add the user-site packages to the path manually
import site
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)
    print(f"Added {user_site} to system path.")

In [ ]:
import os
from langchain_groq import ChatGroq

# 1. Setup the "Voice" (LLM)
os.environ["GROQ_API_KEY"] = "PASTE_YOUR_KEY_HERE"
llm = ChatGroq(
    model_name="llama-3.1-8b-instant", 
    temperature=0
)

# 2. Define the "Analyst" Function
def credi_trust_rag(question):
    # A. Retrieval: Find the top 5 complaints
    docs = retriever.invoke(question)
    
    # B. Context Building: Combine the complaints into one string
    context_text = "\n\n".join([f"Complaint: {d.page_content}" for d in docs])
    
    # C. Prompting: Create the instructions for the AI
    prompt = f"""You are a financial analyst for CrediTrust. 
    Answer the question based ONLY on the complaints provided below.
    If the answer isn't there, say you don't have enough information.

    COMPLAINTS:
    {context_text}

    QUESTION: 
    {question}

    ANSWER:"""
    
    # D. Generation: Send to the LLM
    response = llm.invoke(prompt)
    
    return response.content, docs

print("✅ Manual RAG Function is ready!")

✅ Manual RAG Function is ready!


In [ ]:
import os
import torch
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. LOAD THE RETRIEVER (Same as before)
local_embed_path = "../models/all-MiniLM-L6-v2/"
embeddings = HuggingFaceEmbeddings(model_name=local_embed_path)
vector_db = FAISS.load_local("../vector_store/complaints_index", embeddings, allow_dangerous_deserialization=True)
retriever = vector_db.as_retriever(search_kwargs={"k": 3}) # K=3 is better for small models

# 2. LOAD THE LOCAL LLM (The "Generator")
local_llm_path = "../models/smollm-135m/"
print("Loading local LLM... this may take a moment.")

tokenizer = AutoTokenizer.from_pretrained(local_llm_path)
model = AutoModelForCausalLM.from_pretrained(local_llm_path)

# Create a local generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device="cpu", 
    max_new_tokens=150,
    temperature=0.1,
    do_sample=True
)

# 3. THE RAG FUNCTION
def credi_trust_rag_local(question):
    # Retrieve
    docs = retriever.invoke(question)
    context_text = "\n".join([d.page_content for d in docs])
    
    # Prompting for SmolLM (Instruction format)
    prompt = f"<|im_start|>system\nYou are a financial analyst. Answer the question using ONLY the context provided.<|im_end|>\n<|im_start|>user\nContext: {context_text}\n\nQuestion: {question}<|im_end|>\n<|im_start|>assistant\n"
    
    # Generate
    output = generator(prompt)
    answer = output[0]['generated_text'].split("<|im_start|>assistant\n")[-1]
    
    return answer.strip(), docs

# 4. RUN FINAL EVALUATION
test_questions = [
    "What are the biggest complaints regarding credit card fees?",
    "Why are people unhappy with their Savings Accounts?",
    "Do customers mention 'hidden fees'?"
]

results = []
for q in test_questions:
    print(f"Analyzing: {q}...")
    ans, sources = credi_trust_rag_local(q)
    results.append({"Question": q, "Answer": ans})

df_eval = pd.DataFrame(results)
print("\n✅ TASK 3 COMPLETE (OFFLINE MODE)!")
df_eval

c:\Users\DELL 7020\Desktop\rag-complaint-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\DELL 7020\AppData\Local\Temp\ipykernel_16004\2438573638.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4506.66it/s]


Loading local LLM... this may take a moment.


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2876.61it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Analyzing: What are the biggest complaints regarding credit card fees?...


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analyzing: Why are people unhappy with their Savings Accounts?...


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analyzing: Do customers mention 'hidden fees'?...

✅ TASK 3 COMPLETE (OFFLINE MODE)!


,Question,Answer
0,What are the biggest complaints regarding cred...,The biggest complaints regarding credit card f...
1,Why are people unhappy with their Savings Acco...,People are unhappy with their Savings Accounts...
2,Do customers mention 'hidden fees'?,"Based on the context provided, it is not expli..."
